# **Image Embeddings**


This notebook analyzes image embeddings generated by multiple vision transformer models to explore similarity and feature distributions across a custom dataset. Monitoring shifts in the embedding space is important because differences in learned representations can impact retrieval performance, clustering, and model interpretability.

The analysis focuses on embedding vectors extracted from models such as ViT, DeiT, CvT, Swin, PVT, MaxViT, and T2T-ViT to compare how each model represents images in the dataset. Two complementary approaches are used to quantify similarity and structure:


*   **t-SNE Visualization**: A dimensionality reduction technique that projects high-dimensional embeddings into 2D space, allowing intuitive inspection of clusters and feature relationships.

*  **Top-3 Similarity Retrieval**: Measures the nearest neighbors for a query image in the embedding space using Euclidean or cosine distance, providing a practical view of how well models capture similarity.








Additionally, the notebook visualizes t-SNE embeddings and top-3 retrieved images to allow interactive exploration and comparison across models.

**Objectives of the analysis:**

* Generate embeddings for each image using multiple transformer-based models.
* Reduce dimensionality of embeddings with t-SNE for visualization.
* Compare model representations through similarity searches.
* Quantify and visually inspect clustering and similarity patterns.
* Provide an interactive interface for exploring embeddings and retrieved images.

This approach provides a systematic way to understand transformer embeddings, compare model behaviors, and evaluate image similarity performance in a dataset.




**Install Required Packages**

In [ ]:
# Install all necessary libraries for deep learning, computer vision, and interactive demos
!pip install timm transformers gradio scikit-learn matplotlib pillow torchvision
# timm        -> pre-trained image models for computer vision
# transformers-> Hugging Face library for NLP and vision transformers
# gradio      -> create web interfaces for ML models
# scikit-learn-> general machine learning tools (preprocessing, metrics, classifiers)
# matplotlib  -> data and image visualization
# pillow      -> image processing (open, manipulate, save images)
# torchvision -> PyTorch library for vision (datasets, models, transforms)

In [ ]:
# Clone the T2T-ViT repository from GitHub
# T2T-ViT stands for "Tokens-to-Token Vision Transformer"
# This repository contains code, models, and scripts for training and using T2T-ViT for computer vision tasks.
!git clone https://github.com/yitu-opensource/T2T-ViT.git

Cloning into 'T2T-ViT'...
remote: Enumerating objects: 213, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 213 (delta 72), reused 43 (delta 38), pack-reused 128 (from 1)
Receiving objects: 100% (213/213), 15.53 MiB | 14.34 MiB/s, done.
Resolving deltas: 100% (122/122), done.


**Import Required Libraries**

In [ ]:
# Python built-in libraries
import os               # Operating system utilities (file paths, directory management)
import sys              # System-specific parameters and functions

# PyTorch and related libraries
import torch            # Core PyTorch library for tensors and deep learning
import timm             # PyTorch Image Models library for pre-trained vision models
from torchvision import transforms  # Image preprocessing utilities (resize, normalize, augmentations)

# Scientific computing and ML
import numpy as np      # Numerical computations and array operations
from sklearn.manifold import TSNE               # Dimensionality reduction for visualization (t-SNE)
from sklearn.metrics.pairwise import euclidean_distances  # Compute pairwise Euclidean distances
from sklearn.preprocessing import LabelEncoder  # Encode categorical labels as integers

# Image processing and visualization
import matplotlib.pyplot as plt  # Plotting library for graphs, images, and results
from PIL import Image             # Python Imaging Library (open, manipulate, save images)

# Web-based ML demo
import gradio as gr  # Create interactive UIs to demo ML models in a browser

# Hugging Face Transformers: Vision Transformer models and processors
from transformers import (
    ViTModel,            # Vanilla Vision Transformer model
    ViTImageProcessor,   # Preprocessing utilities for ViT inputs
    DeiTModel,           # Data-efficient Image Transformer model
    DeiTImageProcessor,  # Preprocessing utilities for DeiT inputs
    CvtModel,            # Convolutional vision transformer model
    AutoImageProcessor   # Automatically selects the correct processor for a given model
)

**T2T-ViT Setup**

In [ ]:
# Add the cloned T2T-ViT repository to Python's module search path
# This allows you to import files from the repo as if they were installed packages
sys.path.append("/content/T2T-ViT")  # Adjust the path if you cloned somewhere else

# Import the specific T2T-ViT model architecture
# t2t_vit_14 is the "Tokens-to-Token Vision Transformer" with 14 layers
from models.t2t_vit import t2t_vit_14

/usr/local/lib/python3.12/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


**Folder Setup**

In [ ]:
# Import PyTorch library (used for deep learning and tensor operations)
import torch

# Import image transformation utilities from torchvision
from torchvision import transforms

# Import Path for handling file system paths in a clean, object-oriented way
from pathlib import Path

# Import files module from Google Colab to upload files interactively
from google.colab import files

# Import zipfile module to handle ZIP file extraction
import zipfile

# Import os module for interacting with the operating system
import os


# Automatically select device:
# If GPU (CUDA) is available → use it (faster computation)
# Otherwise → fall back to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"


# Define a sequence of image preprocessing steps
transform = transforms.Compose([

    # Resize every image to 224x224 pixels
    # This is required because most deep learning models (like ViT, ResNet)
    # expect fixed-size inputs
    transforms.Resize((224, 224)),

    # Convert image (PIL format) into a PyTorch tensor
    # Also scales pixel values from [0, 255] → [0, 1]
    transforms.ToTensor()

])


# --- User Folder Upload Section ---

# Open a file upload dialog in Google Colab
# User is expected to upload a ZIP file containing images
uploaded = files.upload()


# Get the name of the uploaded file
# uploaded is a dictionary → {filename: file_content}
# next(iter(...)) extracts the first uploaded file name
zip_path = next(iter(uploaded))


# Define a folder path where extracted images will be stored
DATA_FOLDER = Path("/content/uploaded_images")


# Check if the folder already exists
if not DATA_FOLDER.exists():

    # If not, create the folder (including parent directories if needed)
    DATA_FOLDER.mkdir(parents=True)


# Open the uploaded ZIP file in read mode
with zipfile.ZipFile(zip_path, 'r') as zip_ref:

    # Extract all files from the ZIP into the target folder
    zip_ref.extractall(DATA_FOLDER)


# Print confirmation message showing where images are stored
print(f"Images uploaded and extracted to: {DATA_FOLDER}")

Saving classes.zip to classes.zip
Images uploaded and extracted to: /content/uploaded_images


**Load Images**

In [ ]:
# Create empty lists to store:
# 1. Full paths of images
# 2. Corresponding labels (categories)
image_paths = []
labels = []


# Loop through all files present in the DATA_FOLDER directory
for f in os.listdir(DATA_FOLDER):

    # Convert filename to lowercase and check if it ends with valid image extensions
    # This ensures only image files are processed (ignores other files like .txt, .csv, etc.)
    if f.lower().endswith((".jpg", ".png", ".jpeg", ".jfif")):

        # Construct full file path by joining folder path and file name
        # Example: "/content/uploaded_images/cat_001.jpg"
        image_paths.append(os.path.join(DATA_FOLDER, f))


        # Extract label from file name
        # Assumption: filename format is "label_something.jpg"
        # Example:
        #   "cat_001.jpg" → split("_") → ["cat", "001.jpg"]
        #   Taking first part → "cat"
        labels.append(f.split("_")[0])


# Print summary:
# Total number of images processed
# Total number of unique labels (classes)
print(f"Loaded {len(image_paths)} images with {len(set(labels))} unique labels.")

Loaded 45 images with 9 unique labels.


**Encode Labels**

In [ ]:
# Initialize a LabelEncoder
# This converts categorical string labels (like "cat", "dog")
# into numeric form (like 0, 1, 2) which models can understand
label_encoder = LabelEncoder()


# Fit the encoder on all labels and transform them into integers
# - fit(): learns all unique classes (e.g., ["cat", "dog"])
# - transform(): assigns a number to each label
# Example:
# ["cat", "dog", "cat"] → [0, 1, 0]
label_ids = label_encoder.fit_transform(labels)


# --- Load Images ---
# Create an empty list to store loaded image objects
dataset_images = []


# Loop through each image path collected earlier
for path in image_paths:

    # Open the image file from disk
    img = Image.open(path)

    # Convert image to RGB format
    # This ensures:
    # - All images have 3 channels (Red, Green, Blue)
    # - Avoids issues with grayscale (1 channel) or RGBA (4 channels)
    img = img.convert("RGB")

    # Add the processed image to the dataset list
    dataset_images.append(img)


# Print total number of images successfully loaded
print("Images:", len(dataset_images))

Images: 45


**Load Models**

In [ ]:
# --- Hugging Face Vision Transformers ---

# Load a pre-trained Vision Transformer (ViT) model
# "google/vit-base-patch16-224":
# - Base model size
# - Patch size = 16x16
# - Input image size = 224x224
vit_model = ViTModel.from_pretrained("google/vit-base-patch16-224").to(device)

# Load the corresponding image processor for ViT
# This handles preprocessing like resizing, normalization, etc.
vit_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")


# Load a pre-trained DeiT model (Data-efficient Image Transformer)
# Similar to ViT but trained more efficiently with less data
deit_model = DeiTModel.from_pretrained("facebook/deit-base-patch16-224").to(device)

# Load the corresponding processor for DeiT
deit_processor = DeiTImageProcessor.from_pretrained("facebook/deit-base-patch16-224")


# Load a pre-trained CvT model (Convolutional Vision Transformer)
# Combines CNN + Transformer ideas (better local + global features)
cvt_model = CvtModel.from_pretrained("microsoft/cvt-13").to(device)

# Load a generic image processor for CvT
# AutoImageProcessor automatically selects the correct preprocessing steps
cvt_processor = AutoImageProcessor.from_pretrained("microsoft/cvt-13")


# --- TIMM Models (PyTorch Image Models Library) ---

# Create Swin Transformer model
# "swin_base_patch4_window7_224":
# - Uses shifted windows for efficient attention
swin = timm.create_model("swin_base_patch4_window7_224", pretrained=True)

# Create Pyramid Vision Transformer (PVT)
# Builds hierarchical feature maps (like CNNs but transformer-based)
pvt = timm.create_model("pvt_v2_b2", pretrained=True)

# Create MaxViT model
# Combines convolution + block/grid attention
maxvit = timm.create_model("maxvit_tiny_tf_224", pretrained=True)


# --- Prepare TIMM Models for Feature Extraction ---

# Loop through all TIMM models
for m in [swin, pvt, maxvit]:

    # Remove the final classification layer (head)
    # Setting it to 0 means we only want feature embeddings, not class predictions
    m.reset_classifier(0)

    # Move model to selected device (GPU if available, else CPU)
    m.to(device)

    # Set model to evaluation mode
    # This disables:
    # - Dropout
    # - BatchNorm training behavior
    # Ensures consistent outputs during inference
    m.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

The image processor of type `ConvNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/101M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/124M [00:00<?, ?B/s]

In [ ]:
# --- T2T-ViT Model ---

# Initialize the T2T-ViT (Tokens-to-Token Vision Transformer) model
# "t2t_vit_14" → variant with 14 transformer layers
# pretrained=False → model starts with random weights (not trained on ImageNet)
# This means:
# - It will NOT give meaningful embeddings unless trained
# - Useful if you plan to train from scratch
t2t = t2t_vit_14(pretrained=False)


# Move the model to the selected device
# If GPU is available → faster computation
# Else → CPU
t2t = t2t.to(device)


# Set the model to evaluation mode
# This is important during inference because:
# - Disables dropout (no randomness)
# - Uses fixed batch normalization statistics
# Ensures consistent and stable outputs
t2t.eval()


# --- Ensure all other transformer models are also in evaluation mode ---

# Set ViT model to evaluation mode
vit_model.eval()   # Vision Transformer

# Set DeiT model to evaluation mode
deit_model.eval()  # Data-efficient Image Transformer

# Set CvT model to evaluation mode
cvt_model.eval()   # Convolutional Vision Transformer

adopt performer encoder for tokens-to-token


CvtModel(
  (encoder): CvtEncoder(
    (stages): ModuleList(
      (0): CvtStage(
        (embedding): CvtEmbeddings(
          (convolution_embeddings): CvtConvEmbeddings(
            (projection): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(2, 2))
            (normalization): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (layers): Sequential(
          (0): CvtLayer(
            (attention): CvtAttention(
              (attention): CvtSelfAttention(
                (convolution_projection_query): CvtSelfAttentionProjection(
                  (convolution_projection): CvtSelfAttentionConvProjection(
                    (convolution): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
                    (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
                  )
                  (linear_pro

**Feature Embeddings**

In [ ]:
# --- Hugging Face Vision Transformer Embeddings ---

def embed_vit(img):
    """
    Get embeddings from ViT (Vision Transformer) for a single image.
    """

    # Preprocess the image using ViT processor
    # - Converts PIL image → tensor
    # - Resizes, normalizes automatically (as required by ViT)
    # - return_tensors="pt" → returns PyTorch tensors
    inputs = vit_processor(images=img, return_tensors="pt").to(device)

    # Disable gradient computation (saves memory + faster inference)
    with torch.no_grad():

        # Forward pass through the ViT model
        # Output contains hidden states of all tokens
        out = vit_model(**inputs)

    # Extract embedding:
    # - out.last_hidden_state shape → (batch_size, num_tokens, hidden_dim)
    # - [:, 0] → selects [CLS] token (global image representation)
    # - move to CPU → convert to numpy → flatten to 1D vector
    return out.last_hidden_state[:, 0].cpu().numpy().flatten()



def embed_deit(img):
    """
    Get embeddings from DeiT (Data-efficient Image Transformer) for a single image.
    """

    # Preprocess image using DeiT-specific processor
    inputs = deit_processor(images=img, return_tensors="pt").to(device)

    # Forward pass without gradients
    with torch.no_grad():
        out = deit_model(**inputs)

    # Extract [CLS] token embedding (same logic as ViT)
    return out.last_hidden_state[:, 0].cpu().numpy().flatten()



def embed_cvt(img):
    """
    Get embeddings from CvT (Convolutional Vision Transformer) for a single image.
    """

    # Preprocess image using CvT processor
    inputs = cvt_processor(images=img, return_tensors="pt").to(device)

    # Forward pass
    with torch.no_grad():
        out = cvt_model(**inputs)

    # Instead of using [CLS], take mean of all tokens
    # Why?
    # - CvT may not rely strongly on CLS token
    # - Mean pooling gives a more stable global representation
    return out.last_hidden_state.mean(dim=1).cpu().numpy().flatten()



# --- TIMM Model Embeddings ---

def embed_timm(model, img):
    """
    Get embeddings from a TIMM model for a single image.
    """

    # Apply manual preprocessing:
    # - transform(img) → resize + convert to tensor
    # - unsqueeze(0) → add batch dimension → shape becomes (1, C, H, W)
    x = transform(img).unsqueeze(0).to(device)

    # Forward pass without gradients
    with torch.no_grad():

        # Pass image through model
        # Since classifier head is removed → returns feature embedding
        emb = model(x)

    # Move to CPU → convert to numpy → flatten to 1D vector
    return emb.cpu().numpy().flatten()

**Initialize Embeddings**

In [ ]:
# Dictionary to store feature embeddings for each model
# Keys correspond to model names, values are empty lists that will hold embeddings
all_embeddings = {
    "vit": [],     # Vision Transformer embeddings
    "deit": [],    # Data-efficient Image Transformer embeddings
    "cvt": [],     # Convolutional Vision Transformer embeddings
    "swin": [],    # Swin Transformer embeddings (TIMM)
    "pvt": [],     # PVT Transformer embeddings (TIMM)
    "maxvit": [],  # MaxViT embeddings (TIMM)
    "t2t": []      # Tokens-to-Token Vision Transformer embeddings
}

# Each list will eventually hold 1D numpy arrays (flattened feature vectors)
# This structure makes it easy to organize embeddings by model for visualization, clustering, or retrieval

**Generate Embeddings**

In [ ]:
# Loop through all loaded images (each is a PIL image)
for img in dataset_images:

    # --- Hugging Face Models ---

    # Generate embedding using ViT and store it
    # embed_vit(img) → returns a 1D vector
    all_embeddings["vit"].append(embed_vit(img))      # ViT

    # Generate embedding using DeiT and store it
    all_embeddings["deit"].append(embed_deit(img))    # DeiT

    # Generate embedding using CvT and store it
    all_embeddings["cvt"].append(embed_cvt(img))      # CvT


    # --- TIMM Models ---

    # Generate embedding using Swin Transformer (TIMM)
    # embed_timm handles preprocessing + forward pass
    all_embeddings["swin"].append(embed_timm(swin, img))      # Swin Transformer

    # Generate embedding using Pyramid Vision Transformer (PVT)
    all_embeddings["pvt"].append(embed_timm(pvt, img))        # PVT

    # Generate embedding using MaxViT
    all_embeddings["maxvit"].append(embed_timm(maxvit, img))  # MaxViT

    # Generate embedding using T2T-ViT
    # Note: This model is not pretrained (earlier), so embeddings may not be meaningful
    all_embeddings["t2t"].append(embed_timm(t2t, img))        # T2T-ViT


# --- Convert lists to numpy arrays ---

# Loop through each model key in the dictionary
for k in all_embeddings:

    # Convert list of embeddings → numpy array
    # Before: list of vectors
    # After: 2D array (num_images, feature_dim)
    all_embeddings[k] = np.array(all_embeddings[k])


# --- Explanation (what we achieved) ---

# After this:
# all_embeddings["vit"]   → shape: (num_images, feature_dim_vit)
# all_embeddings["deit"]  → shape: (num_images, feature_dim_deit)
# all_embeddings["cvt"]   → shape: (num_images, feature_dim_cvt)
# ...
#
# Each row = one image
# Each column = feature dimension
#
# This format is ideal for:
# - Similarity search (cosine similarity, etc.)
# - Clustering (KMeans, DBSCAN)
# - Visualization (t-SNE, PCA)

**t-SNE Embeddings**

In [ ]:
from sklearn.manifold import TSNE

# Dictionary to store 2D t-SNE projections for each model
tsne_results = {}

# Loop over each model's embeddings
for name in all_embeddings:

    # Initialize t-SNE with 2 components (for 2D visualization)
    # - random_state ensures reproducibility
    # - perplexity balances attention between local and global structure
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)

    # Fit t-SNE on the embeddings and transform them to 2D
    tsne_results[name] = tsne.fit_transform(all_embeddings[name])

# Explanation:
# - tsne_results[name] is a numpy array of shape (num_images, 2)
#   suitable for plotting with matplotlib or seaborn.
# - Each key corresponds to a different model's embedding space.
# - Perplexity ~30 works well for medium-sized datasets (10s–100s of images).

**Plot t-SNE**

In [ ]:
# Import matplotlib's pyplot module for plotting graphs
import matplotlib.pyplot as plt

# Import matplotlib color utilities (not directly used here but useful for custom colors)
import matplotlib.colors as mcolors


# Define a function to plot t-SNE results for a given model
def plot_model(name, query_point=None, coords=None):

    """
    Plot 2D t-SNE projections for a given model.

    Parameters:
    - name: str
        Name of the model (used as key to access tsne_results dictionary)

    - query_point: optional
        A single 2D point (x, y) to highlight on the plot
        Typically used to mark a specific image

    - coords: optional
        Precomputed 2D coordinates
        If not provided, function will use tsne_results[name]

    Returns:
    - fig: matplotlib figure object
    """


    # Check if coordinates are NOT provided by the user
    if coords is None:

        # If coords is None, fetch the t-SNE result corresponding to the model name
        # tsne_results[name] → numpy array of shape (num_images, 2)
        coords = tsne_results[name]


    # Create a new figure and axes object
    # figsize=(5,5) → square plot with width=5 inches and height=5 inches
    fig, ax = plt.subplots(figsize=(5, 5))


    # Plot all points using a scatter plot
    ax.scatter(

        # coords[:, 0] → all x-coordinates (first column of t-SNE output)
        coords[:, 0],

        # coords[:, 1] → all y-coordinates (second column of t-SNE output)
        coords[:, 1],

        # c=label_ids → assigns a color to each point based on its label
        # label_ids is a list/array of integers (e.g., 0,1,2,...)
        # Same label → same color
        c=label_ids,

        # cmap="tab10" → predefined colormap with 10 distinct colors
        # Used for categorical data visualization
        cmap="tab10",

        # s=40 → size of each scatter point
        # Larger value → bigger points
        s=40
    )


    # Check if a query point is provided
    if query_point is not None:

        # Plot the query point separately so it stands out
        ax.scatter(

            # query_point[0] → x-coordinate of the query point
            query_point[0],

            # query_point[1] → y-coordinate of the query point
            query_point[1],

            # marker="X" → use 'X' shape instead of default circle
            marker="X",

            # color="black" → make it visually distinct from other points
            color="black",

            # s=200 → much larger size to highlight clearly
            s=200
        )


    # Set the title of the plot
    # name.upper() → converts model name to uppercase (e.g., "vit" → "VIT")
    ax.set_title(name.upper())


    # Return the figure object instead of directly displaying it
    # This allows:
    # - saving the plot later
    # - modifying it outside the function
    return fig

**Legend Plot**

In [ ]:
# Import module to create colored legend boxes (patches)
import matplotlib.patches as mpatches

# Import color normalization utilities
import matplotlib.colors as mcolors

# Import plotting library
import matplotlib.pyplot as plt


# Define a function to generate a legend figure
def legend_plot():

    """
    Create a legend figure for class labels using the same colors as t-SNE plots.

    Returns:
    - fig: matplotlib figure containing the legend
    """


    # Create a new figure and axis
    # figsize=(8,2) → wide and short figure (good for horizontal legend layout)
    fig, ax = plt.subplots(figsize=(8, 2))


    # Select the same colormap used in t-SNE plots
    # plt.cm.tab10 → categorical colormap with 10 distinct colors
    cmap = plt.cm.tab10


    # Normalize label indices to map them into the colormap range
    # vmin=0 → lowest class index
    # vmax=len(label_encoder.classes_)-1 → highest class index
    # This ensures each class gets a unique color
    norm = mcolors.Normalize(
        vmin=0,
        vmax=len(label_encoder.classes_) - 1
    )


    # Initialize an empty list to store legend items (patches)
    patches = []


    # Loop through each class label with its index
    for i, label in enumerate(label_encoder.classes_):

        # Create a colored patch for each class
        patches.append(

            mpatches.Patch(

                # Assign color based on normalized index
                # cmap(norm(i)) → converts class index to color
                color=cmap(norm(i)),

                # Set label text (class name)
                label=label
            )
        )


    # Add legend to the axis
    ax.legend(

        # handles=patches → list of colored boxes with labels
        handles=patches,

        # loc="center" → place legend at the center of the figure
        loc="center",

        # ncol=min(6, len(patches)) →
        # Number of columns in legend:
        # - Max 6 columns
        # - If fewer classes → use fewer columns
        ncol=min(6, len(patches)),

        # Title displayed above legend
        title="Classes"
    )


    # Turn off axis lines, ticks, and labels
    # Because we only want the legend (no graph)
    ax.axis("off")


    # Return the figure object for further use (display/save)
    return fig

**Embed Query**

In [ ]:
# These functions are designed to generate embeddings
# from multiple vision transformer models and to
# visualize their architecture or performance.
# Each function call leverages a different model variant
# to allow comparisons across embeddings and model behaviors.

# -----------------------------------------------
# Function: embed_query
# -----------------------------------------------
# Purpose:
# Takes an input image and generates embeddings using
# several vision transformer (ViT) variants.
# Returns a dictionary where keys are model names
# and values are their corresponding embeddings.
def embed_query(img):
    q = {}

    # Embeddings from standard ViT models
    q["vit"] = embed_vit(img)       # Vanilla Vision Transformer
    q["deit"] = embed_deit(img)     # Data-efficient Image Transformer
    q["cvt"] = embed_cvt(img)       # Convolutional Vision Transformer

    # Embeddings using models integrated via timm library
    q["swin"] = embed_timm(swin, img)      # Swin Transformer (hierarchical)
    q["pvt"] = embed_timm(pvt, img)        # Pyramid Vision Transformer
    q["maxvit"] = embed_timm(maxvit, img)  # MaxViT, combines local and global attention
    q["t2t"] = embed_timm(t2t, img)        # Tokens-to-Token Vision Transformer

    return q

# -----------------------------------------------
# Function: initial_plots
# -----------------------------------------------
# Purpose:
# Generates initial plots or visualizations for each
# vision transformer model. This is useful for
# understanding model architectures or comparing
# model performance.
# Returns a list of plots corresponding to each model.
def initial_plots():
    return [
        plot_model("vit"),      # Plot for Vanilla ViT
        plot_model("deit"),     # Plot for DeiT
        plot_model("cvt"),      # Plot for CvT
        plot_model("swin"),     # Plot for Swin Transformer
        plot_model("pvt"),      # Plot for Pyramid Vision Transformer
        plot_model("maxvit"),   # Plot for MaxViT
        plot_model("t2t")       # Plot for Tokens-to-Token Transformer
    ]

***Retrieval Functions***

In [ ]:
# These functions find the top-3 most similar items
# from a dataset given a query embedding.
# One uses cosine similarity (angle-based similarity)
# and the other uses Euclidean distance (vector magnitude).

# -----------------------------------------------
# Function: get_top3_cosine
# -----------------------------------------------
# Purpose:
# Given a query embedding and a dataset of embeddings,
# returns the indices of the top-3 most similar items
# based on cosine similarity.
# Cosine similarity measures the angle between vectors,
# ignoring their magnitude.
def get_top3_cosine(query_emb, dataset_emb):
    # Compute cosine similarity between the query and all dataset embeddings
    sims = cosine_similarity([query_emb], dataset_emb)[0]

    # Get indices of top-3 highest similarity scores
    top3_idx = sims.argsort()[::-1][:3]  # descending order
    return top3_idx

# -----------------------------------------------
# Function: get_top3
# -----------------------------------------------
# Purpose:
# Given a query embedding and a dataset of embeddings,
# returns the indices of the top-3 closest items
# based on Euclidean distance.
# Euclidean distance measures the straight-line distance
# between vectors in feature space.
def get_top3(query_emb, dataset_emb):
    # Compute Euclidean distances between query and dataset embeddings
    dists = euclidean_distances([query_emb], dataset_emb)[0]

    # Get indices of the three smallest distances
    return dists.argsort()[:3]  # ascending order (closest first)

**Search**

In [ ]:
# Given an input image, this function performs a multi-model
# search across precomputed embeddings for several vision
# transformer models. It generates both t-SNE plots for
# visualization and top-3 nearest neighbors for each model.

def search(img):
    # -----------------------------------------------
    # Step 1: Generate embeddings for the query image
    # -----------------------------------------------
    # Use multiple vision transformer models to embed the image
    q = embed_query(img)

    # List of all model names to iterate over
    model_names = ["vit", "deit", "cvt", "swin", "pvt", "maxvit", "t2t"]

    # Containers for plots and top-3 results
    plots = []
    galleries = []

    # -----------------------------------------------
    # Step 2: Process each model separately
    # -----------------------------------------------
    for m in model_names:
        # Retrieve precomputed dataset embeddings for this model
        dataset_emb = all_embeddings[m]

        # Get the embedding for the query image for this model
        query_emb = q[m]

        # -----------------------------------------------
        # Step 2a: Combine query and dataset embeddings
        # for t-SNE visualization
        # -----------------------------------------------
        combined = np.vstack([dataset_emb, query_emb])

        # Recompute t-SNE for 2D visualization
        tsne = TSNE(n_components=2, random_state=42, perplexity=30)
        coords = tsne.fit_transform(combined)

        # Split back into dataset coordinates and query coordinate
        dataset_coords = coords[:-1]
        query_coord = coords[-1]

        # Generate plot for this model showing query location
        plots.append(plot_model(m, query_coord, dataset_coords))

        # -----------------------------------------------
        # Step 2b: Retrieve top-3 nearest neighbors
        # -----------------------------------------------
        idx = get_top3(query_emb, dataset_emb)  # using Euclidean distance

        res = []
        for i in idx:
            # Append image and label tuple
            res.append((dataset_images[i], labels[i]))

        galleries.append(res)

    # -----------------------------------------------
    # Step 3: Return combined visualization and results
    # -----------------------------------------------
    # First the plots, then the top-3 galleries for each model
    return plots + galleries

**Gradio Interface**

In [ ]:
import gradio as gr
import base64
from io import BytesIO

# -----------------------------------------------
# Convert PIL image → base64
# -----------------------------------------------
def pil_to_base64(img):
    buffer = BytesIO()
    img.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode()

# -----------------------------------------------
# Render images with title
# -----------------------------------------------
def render_images(images, model_name):
    html = f"""
    <div style="margin-bottom:20px;">
        <h4 style="margin-bottom:8px; font-weight:600;">{model_name}</h4>
        <div style="display:flex; gap:10px;">
    """

    for img, label in images:
        img_base64 = pil_to_base64(img)

        html += f"""
        <div style="text-align:center;">
            <img src="data:image/png;base64,{img_base64}"
                 style="width:120px; height:120px; object-fit:cover; border-radius:8px;">
            <div style="font-size:12px;">{label}</div>
        </div>
        """

    html += "</div></div>"
    return html

# -----------------------------------------------
# APP
# -----------------------------------------------
with gr.Blocks() as demo:

    gr.Markdown("# Image Embeddings")

    # -----------------------------------------------
    # Legend
    # -----------------------------------------------
    gr.Markdown("## Dataset Class Legend")
    legend = gr.Plot()
    demo.load(fn=legend_plot, outputs=legend)

    # -----------------------------------------------
    # t-SNE (same order)
    # -----------------------------------------------
    gr.Markdown("## t-SNE Visualization")

    # Row 1
    with gr.Row():
        vit_plot = gr.Plot(label="ViT")
        swin_plot = gr.Plot(label="Swin")
        maxvit_plot = gr.Plot(label="MaxViT")

    # Row 2
    with gr.Row():
        deit_plot = gr.Plot(label="DeiT")
        cvt_plot = gr.Plot(label="CvT")
        pvt_plot = gr.Plot(label="PVT")

    # Row 3
    with gr.Row():
        t2t_plot = gr.Plot(label="T2T-ViT")

    demo.load(
        fn=initial_plots,
        outputs=[
            vit_plot,
            swin_plot,
            maxvit_plot,
            deit_plot,
            cvt_plot,
            pvt_plot,
            t2t_plot
        ]
    )

    gr.Markdown("---")

    # -----------------------------------------------
    # Input
    # -----------------------------------------------
    gr.Markdown("## Image Retrieval")
    image_input = gr.Image(type="pil")
    search_btn = gr.Button("Retrieve")

    # -----------------------------------------------
    # OUTPUT (REORDERED)
    # -----------------------------------------------
    gr.Markdown("## Top-3 Similar Images per Model")

    # Row 1
    with gr.Row():
        vit_gallery = gr.HTML()
        swin_gallery = gr.HTML()
        maxvit_gallery = gr.HTML()

    # Row 2
    with gr.Row():
        deit_gallery = gr.HTML()
        cvt_gallery = gr.HTML()
        pvt_gallery = gr.HTML()

    # Row 3
    with gr.Row():
        t2t_gallery = gr.HTML()

    # -----------------------------------------------
    # Wrapper
    # -----------------------------------------------
    def wrapped_search(img):
        results = search(img)

        return [
            results[0],  # vit_plot
            results[3],  # swin_plot
            results[5],  # maxvit_plot
            results[1],  # deit_plot
            results[2],  # cvt_plot
            results[4],  # pvt_plot
            results[6],  # t2t_plot

            render_images(results[7], "ViT"),
            render_images(results[10], "Swin"),
            render_images(results[12], "MaxViT"),
            render_images(results[8], "DeiT"),
            render_images(results[9], "CvT"),
            render_images(results[11], "PVT"),
            render_images(results[13], "T2T-ViT"),
        ]

    # -----------------------------------------------
    # Button Action
    # -----------------------------------------------
    search_btn.click(
        fn=wrapped_search,
        inputs=image_input,
        outputs=[
            vit_plot,
            swin_plot,
            maxvit_plot,
            deit_plot,
            cvt_plot,
            pvt_plot,
            t2t_plot,
            vit_gallery,
            swin_gallery,
            maxvit_gallery,
            deit_gallery,
            cvt_gallery,
            pvt_gallery,
            t2t_gallery
        ]
    )

# -----------------------------------------------
# Launch
# -----------------------------------------------
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://30372f9244fee6f32e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://30372f9244fee6f32e.gradio.live
